In [ ]:
import pandas as pd

In [2]:
titantic_df = pd.read_csv("data/titanic.csv")
titantic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
titantic_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


- Name, Ticket, Cabin doesnt add any value to model as everyone have different values
- Embarked, Sex are nominal(no hierarchy) and Pclass is categorical but already converted

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

nominal_cols = ['Sex', 'Embarked']
numerical_cols = ['Age', 'SibSp', 'Parch', 'Fare']
cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Survived']

preprocessor = ColumnTransformer(transformers=[
    ('Nom', 
        Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(drop='first')), 
        ]), nominal_cols),

    ('Standard', 
        Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_cols),

], remainder='passthrough')

X = titantic_df.drop(columns=cols_to_drop)
y = titantic_df['Survived']

In [43]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

def run_pipeline(model, preprocessor):
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])
    return pipe

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

def run_model(model, preprocessor, X_train, X_val, y_train, y_val):
    pipe = run_pipeline(model, preprocessor)


    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)

    print(f"Accuracy: {accuracy:.2f}")
    print(f"Classification Matrix: \n{classification_report(y_val, y_pred)}\n")
    print(f"Confusion Matrix: \n{confusion_matrix(y_val, y_pred)}")

    return pipe


### Logistic

In [21]:
from sklearn.linear_model import LogisticRegression

run_model(
    LogisticRegression(), 
    preprocessor,
    X_train, X_val, y_train, y_val
)

Accuracy: 0.81
Classification Matrix: 
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179


Confusion Matrix: 
[[90 15]
 [19 55]]


### RandomForest

In [22]:
from sklearn.ensemble import RandomForestClassifier

run_model(
    RandomForestClassifier(n_estimators=200), 
    preprocessor,
    X_train, X_val, y_train, y_val
)

Accuracy: 0.81
Classification Matrix: 
              precision    recall  f1-score   support

           0       0.84      0.84      0.84       105
           1       0.77      0.77      0.77        74

    accuracy                           0.81       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179


Confusion Matrix: 
[[88 17]
 [17 57]]


### tune

In [23]:
from sklearn.model_selection import GridSearchCV

params = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5, 10],
}

pipe = run_pipeline(RandomForestClassifier(), preprocessor)

grid_search = GridSearchCV(pipe, param_grid=params, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'model__max_depth': 5, 'model__min_samples_split': 5, 'model__n_estimators': 100}
0.8300403821530582


In [25]:
run_model(
    RandomForestClassifier(max_depth = 5, min_samples_split = 5, n_estimators = 100), 
    preprocessor,
    X_train, X_val, y_train, y_val
)

Accuracy: 0.82
Classification Matrix: 
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       105
           1       0.84      0.69      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.81       179


Confusion Matrix: 
[[95 10]
 [23 51]]


In [28]:
titantic_df['title'] = titantic_df['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)

In [33]:
titantic_df['title'].value_counts()

title
Mr        517
Miss      182
Mrs       125
Master     40
Other      27
Name: count, dtype: int64

In [32]:
titantic_df['title'] = titantic_df['title'].apply(lambda title: title if title in ['Mr', 'Miss', 'Mrs', 'Master'] else 'Other')

In [45]:
def prepare_data(df):
    df = df.copy()

    df['title'] = df['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)
    df['title'] = df['title'].apply(
        lambda title: title 
        if title in ['Mr', 'Miss', 'Mrs', 'Master'] else 'Other'
    )

    nominal_cols = ['Sex', 'Embarked', 'title']
    numerical_cols = ['Age', 'SibSp', 'Parch', 'Fare']
    cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Survived']

    preprocessor = ColumnTransformer(transformers=[
        ('Nom', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')), 
            ('ohe', OneHotEncoder(drop='first'))]), nominal_cols),

        ('Standard', Pipeline([
            ('imputer', SimpleImputer(strategy='median')), 
            ('scaler', StandardScaler())]), numerical_cols),
    ], remainder='passthrough')

    X = df.drop(columns=cols_to_drop)
    y = df['Survived']

    return train_test_split(X, y, test_size=0.2, random_state=42), preprocessor

(X_train, X_val, y_train, y_val), preprocessor = prepare_data(titantic_df)


In [36]:
from sklearn.model_selection import GridSearchCV

params = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5, 10],
}

pipe = run_pipeline(RandomForestClassifier(), preprocessor)

grid_search = GridSearchCV(pipe, param_grid=params, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 200}
0.8356347877474637


In [46]:
model = run_model(
    RandomForestClassifier(
        max_depth = 10, min_samples_split = 5, n_estimators = 200, random_state=42
    ), 
    preprocessor,
    X_train, X_val, y_train, y_val
)

Accuracy: 0.84
Classification Matrix: 
              precision    recall  f1-score   support

           0       0.85      0.89      0.87       105
           1       0.83      0.77      0.80        74

    accuracy                           0.84       179
   macro avg       0.84      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179


Confusion Matrix: 
[[93 12]
 [17 57]]


In [53]:
import joblib
from pathlib import Path

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

joblib.dump(model, model_dir / 'model.pkl')

['models\\model.pkl']

In [54]:
titantic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr
